# MEG theta power around button presses

This notebook follows the same button-time conversion, epoch windows, Hann-windowed FFT, 2–5 Hz integration, and trial-baseline dB correction as MEG_powerplot_subject.ipynb.

It first computes each participant's mean baseline-corrected theta power for every MEG sensor, averaging across that participant's valid presses. It then averages those participant means, giving each included participant equal weight. The scalp maps and channel table show the group mean in dB.

The existing subject exclusions are kept. The group includes every ID in sub_ID except IDs in excluded_subjects. Set excluded_subjects to an empty list to include every ID in sub_ID.

import packages for file reading, MEG data handling and plotting.

In [ ]:
import h5py
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

define parameters: edit sub_ID to choose the participants for the group average. Excluded IDs are skipped.

In [ ]:
sub_ID = ['05','06','08','09','10','11','13','14','15','16','17','18','20','21','24','25','28']  # subject IDs
excluded_subjects = ['11','16','21']  # participants with questionable data

theta_freq_band = np.array([2, 5])          # frequency band edges in Hz
press_time_wind = np.array([-0.5, 0.5])    # epoch window around each button press
baseline_time_wind = np.array([-1, 0])     # baseline window before each trial
sens_delay = 2.8                            # offset between cue time and MEG epoch start
n_press_slots = 5                           # number of button presses per trial


# load each participant's files

filename_loader uses the same network-share and file naming pattern as the subject notebook. It reads the behavioral press times and cue durations, converts them to seconds, and loads the MNE epochs.

Subject 16 has 240 behavioral rows but 222 saved MEG epochs. For that exact case, the loader keeps the first 222 rows of the trials-by-press-slots array and drops the trailing rows. Other unexplained row-count mismatches stop with an error.

In [ ]:
def filename_loader(id):
    # id is the subject number as a two-digit string.
    subject_folder = "//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data"
    button_press_folder = "/beh_data/s" + str(id)
    BP1 = subject_folder + button_press_folder + "/behDataMEG.mat"
    BP2 = subject_folder + button_press_folder + "/targetMEG.mat"

    # Subject folder names use a leading zero for single-digit IDs.
    if id.startswith("0"):
        spliced_id = id[1:]
        EMG_file = "/s0" + spliced_id + "/Subject" + spliced_id + "_spm_mne-epo"
    else:
        EMG_file = "/s" + str(id) + "/Subject" + str(id) + "_spm_mne-epo"
    CD_file = subject_folder + EMG_file + ".fif"

    # Load the behavioral press times and press identity codes.
    with h5py.File(BP1, "r") as f:
        timings = f["A/timing"][:]
        presses = f["A/press"][:]

    # Load the duration of each trial's cue.
    with h5py.File(BP2, "r") as f:
        cuedur = f["T/cueDur"][:]

    # Put the arrays into trials-by-press-slots layout.
    timings = timings.T
    PRES = presses.T
    cuedur = cuedur.T

    # Add cue duration to press time and convert milliseconds to seconds.
    PT = (timings + np.tile(cuedur, n_press_slots)) / 1000.0

    # Load the epochs and keep the MEG sensor names for this participant.
    CD = mne.read_epochs(CD_file, preload=True)
    if id == "16" and len(CD) == 222 and PT.shape[0] == 240:
        print("Subject 16: using the first 222 behavioral trial rows.")
        PT = PT[:len(CD), :]
        PRES = PRES[:len(CD), :]
    elif len(CD) != PT.shape[0]:
        raise ValueError(
            "Subject " + id + " has " + str(len(CD))
            + " MEG epochs but " + str(PT.shape[0]) + " behavioral trials"
        )

    MEG_sens = []
    for channel in CD.info["ch_names"]:
        if channel.startswith("M"):
            MEG_sens.append(channel)

    sfreq = float(CD.info["sfreq"])
    return PT, PRES, CD, MEG_sens, sfreq

check the participant list

All IDs in sub_ID are processed except IDs in excluded_subjects. Each subject is loaded and averaged separately before the group average is calculated. The participant-level values are retained in a table for review.

In [ ]:
subjects_to_use = [id for id in sub_ID if id not in excluded_subjects]

if len(subjects_to_use) == 0:
    raise ValueError("No subjects remain after applying excluded_subjects.")

print("Subjects included in the group average:", subjects_to_use)

prepare the press and baseline epochs

The press times are shifted onto one continuous timeline by adding the cumulative duration of previous trials and sens_delay. The code then builds a RawArray from the MEG channels and extracts press epochs from −0.5 to +0.5 seconds and baseline epochs from −1 to 0 seconds.

mean_band_power uses the same calculation as the subject notebook: remove the time-series mean, apply a Hann window, calculate the one-sided FFT power spectral density, and integrate the 2–5 Hz bins. Baseline correction is applied later as 10 × log10(press power / that trial's baseline power).

In [ ]:
def press_time_flattener(pt_arr, n_epochs, trial_time):
    # Add each trial's position on the continuous timeline to its press times.
    n_trials, n_press_slots = pt_arr.shape
    trial_offsets = np.arange(n_trials, dtype=float) * trial_time
    press_times_shifted = pt_arr + sens_delay + trial_offsets[:, None]
    trial_cutoff_times = sens_delay + trial_offsets

    # Keep the trial number for each finite press time.
    valid = np.isfinite(press_times_shifted)
    press_trial_indices = np.broadcast_to(
        np.arange(n_trials)[:, None], (n_trials, n_press_slots)
    )[valid]

    print("Removed", int((~valid).sum()), "missing/non-finite press times")
    return press_times_shifted[valid], trial_cutoff_times, press_trial_indices


def epochs_maker(raw, events, window, sfreq, name):
    # Convert event times in seconds to sample indices.
    samples = np.rint(np.asarray(events, dtype=float) * sfreq).astype(int)

    # MNE event columns are sample index, previous value, and event code.
    event_array = np.column_stack([
        samples,
        np.zeros(len(samples), dtype=int),
        np.ones(len(samples), dtype=int),
    ])

    # Extract the requested time window around each event.
    epoch = mne.Epochs(
        raw,
        event_array,
        event_id={"event": 1},
        tmin=float(window[0]),
        tmax=float(window[1]) - 1.0 / sfreq,
        baseline=None,
        preload=True,
        verbose=False,
    )
    print(name, "epochs:", len(epoch))
    return epoch


def mean_band_power(epochs, band, MEG_sens):
    # Get data in epochs × sensors × time order.
    x = epochs.get_data(picks=MEG_sens)
    fs = float(epochs.info["sfreq"])
    n_times = x.shape[-1]

    # This calculation expects one-second epochs.
    if not np.isclose(n_times / fs, 1.0):
        raise ValueError("Expected a 1-second epoch")

    # Calculate a Hann-windowed, one-sided power spectral density.
    window = np.hanning(n_times)
    x = x - x.mean(axis=-1, keepdims=True)
    fft = np.fft.rfft(x * window, axis=-1)
    psd = (np.abs(fft) ** 2) / (fs * np.sum(window ** 2))
    freqs = np.fft.rfftfreq(n_times, d=1.0 / fs)

    # Integrate the frequency bins within the requested band.
    in_band = (freqs >= band[0]) & (freqs <= band[1])
    band_psd = psd[..., in_band]
    band_freqs = freqs[in_band]
    return np.sum(
        0.5 * (band_psd[..., 1:] + band_psd[..., :-1]) * np.diff(band_freqs),
        axis=-1,
    )

calculate theta power for each participant

Each participant is processed independently using the same steps as the subject notebook. Press power is divided by the baseline power from the same trial, converted to dB, and averaged across valid presses separately for each sensor.

The function returns one value per MEG sensor for each participant. A sensor's group mean is therefore the mean of participant means, rather than a pooled mean that gives participants with more presses greater weight.

In [ ]:
def participant_theta_power(id):
    press_times, presses, data, MEG_sens, sfreq = filename_loader(id)

    # Reshape the MEG epochs and make a continuous-style RawArray.
    MEG_data = data.get_data(picks=MEG_sens)
    n_epochs, n_channels, n_times = MEG_data.shape
    trial_time = n_times / sfreq
    MEG_data = MEG_data.transpose(1, 0, 2).reshape(
        n_channels, n_epochs * n_times
    )

    meg_picks = mne.pick_channels(data.info["ch_names"], include=MEG_sens)
    info = mne.pick_info(data.info, meg_picks, copy=True)
    raw = mne.io.RawArray(MEG_data, info, verbose=False)

    # Shift press times to the concatenated timeline and retain each press's trial.
    press_times_shifted, trial_cutoff_times, press_trial_indices = (
        press_time_flattener(press_times, n_epochs, trial_time)
    )

    press_epochs = epochs_maker(
        raw, press_times_shifted, press_time_wind, sfreq, "press"
    )
    baseline_epochs = epochs_maker(
        raw, trial_cutoff_times, baseline_time_wind, sfreq, "baseline"
    )

    # Calculate 2–5 Hz power for press and baseline epochs.
    theta_power_by_press = mean_band_power(
        press_epochs, theta_freq_band, MEG_sens
    )
    theta_power_baseline = mean_band_power(
        baseline_epochs, theta_freq_band, MEG_sens
    )

    # Match each press epoch to the baseline from the same trial.
    theta_baseline_by_trial = np.full(
        (len(trial_cutoff_times), len(MEG_sens)), np.nan
    )
    theta_baseline_by_trial[baseline_epochs.selection] = theta_power_baseline
    press_trial_ids = press_trial_indices[press_epochs.selection]
    theta_baseline_for_press = theta_baseline_by_trial[press_trial_ids]

    # Keep presses with a usable baseline.
    has_baseline = np.isfinite(theta_baseline_for_press).any(axis=1)
    theta_power_by_press = theta_power_by_press[has_baseline]
    theta_baseline_for_press = theta_baseline_for_press[has_baseline]
    press_times_used = press_times_shifted[press_epochs.selection][has_baseline]

    if len(theta_power_by_press) == 0:
        raise RuntimeError("No presses have both a valid press epoch and baseline")

    # Convert each press-to-baseline power ratio to dB.
    valid_power = (
        np.isfinite(theta_power_by_press)
        & np.isfinite(theta_baseline_for_press)
        & (theta_power_by_press > 0)
        & (theta_baseline_for_press > 0)
    )
    theta_db_by_press = np.full(theta_power_by_press.shape, np.nan)
    power_ratio = np.ones_like(theta_power_by_press)
    np.divide(
        theta_power_by_press,
        theta_baseline_for_press,
        out=power_ratio,
        where=valid_power,
    )
    theta_db_by_press[valid_power] = 10.0 * np.log10(power_ratio[valid_power])

    # Average valid button presses for each MEG sensor.
    sensor_counts = np.isfinite(theta_db_by_press).sum(axis=0)
    theta_db_mean = np.divide(
        np.nansum(theta_db_by_press, axis=0),
        sensor_counts,
        out=np.full(len(MEG_sens), np.nan),
        where=sensor_counts > 0,
    )

    print("Subject", id, "- retained button presses:", len(press_times_used))
    return theta_db_mean, MEG_sens, info, len(press_times_used)

average channel power across participants

The loop collects each participant's per-channel mean dB change. Sensor names are checked so values from different participants cannot be averaged under mismatched channel labels. The group average gives each participant equal weight; NaN values are ignored for a sensor when that participant has no valid power estimate.

In [ ]:
subject_power = []
subjects_used = []
retained_press_counts = []
group_sensors = None
group_info = None

for id in subjects_to_use:
    theta_db_mean, MEG_sens, info, n_presses = participant_theta_power(id)

    if group_sensors is None:
        group_sensors = MEG_sens
        group_info = info
    elif MEG_sens != group_sensors:
        raise ValueError("MEG sensor names or order differ for subject " + id)

    subject_power.append(theta_db_mean)
    subjects_used.append(id)
    retained_press_counts.append(n_presses)

# Rows are participants and columns are MEG sensors.
theta_db_by_subject = pd.DataFrame(
    subject_power, index=subjects_used, columns=group_sensors
)

# Average participant means separately for each sensor.
theta_db_group_mean = theta_db_by_subject.mean(axis=0, skipna=True)
n_participants_by_sensor = theta_db_by_subject.count(axis=0)

participant_summary = pd.DataFrame({
    "subject": subjects_used,
    "retained_press_count": retained_press_counts,
})
channel_summary = pd.DataFrame({
    "MEG_channel": group_sensors,
    "mean_theta_power_change_dB": theta_db_group_mean.to_numpy(),
    "n_participants": n_participants_by_sensor.to_numpy(),
})

display(participant_summary)
display(channel_summary)

plot the group average

The channel table above contains the averaged value for every sensor. The scalp maps show the same group mean, plotted separately for each MEG sensor type.

In [ ]:
theta_db_mean_all = theta_db_group_mean.to_numpy()
sensor_types = [
    mne.channel_type(group_info, index)
    for index in range(len(group_sensors))
]

for sensor_type in sorted(set(sensor_types)):
    picks = np.array([
        index for index, kind in enumerate(sensor_types)
        if kind == sensor_type
    ])
    sensor_info = mne.pick_info(group_info, picks, copy=True)

    fig, ax = plt.subplots(figsize=(7, 6))
    image, _ = mne.viz.plot_topomap(
        theta_db_mean_all[picks],
        sensor_info,
        ch_type=sensor_type,
        axes=ax,
        cmap="RdBu_r",
        contours=0,
        show=False,
    )
    ax.set_title(
        "Mean participant 2–5 Hz theta power change (" + sensor_type + ")"
    )
    fig.colorbar(image, ax=ax, label="Power change (dB)")

plt.show()